# Pyccolo — live in your browser 🎼

[Pyccolo](https://github.com/smacke/pyccolo) (pronounced like the instrument
"piccolo") is a library for **declarative instrumentation** in Python. You say
*what* you want to observe or change — "every float literal", "every attribute
access", "every function call" — and Pyccolo rewrites the source to make it
happen. No bytecode patching, no hand-written `ast.NodeTransformer`.

This notebook runs entirely in your browser via
[JupyterLite](https://jupyterlite.readthedocs.io/) (Pyodide) — nothing is
installed on your machine. Run the cells top-to-bottom.

> **One thing to know:** a program's syntax tree is fixed when its cell is
> compiled, *before* a tracer becomes active. So to instrument code we wrap it
> in `pyc.exec("...")` — that source is transformed with the active tracers in
> effect. (The `sys.settrace`-style `call`/`return_` events are the exception;
> they fire on ordinary in-cell code.)

Start by installing pyccolo — resolved offline from the wheel bundled with this
site:

In [ ]:
%pip install pyccolo

In [ ]:
import pyccolo as pyc
print(pyc.__version__)

## 1. Hello, world — a handler on every statement

A tracer is a subclass of `pyc.BaseTracer`. Decorate a method with an **event**
(here `@pyc.before_stmt`) and it runs whenever that event fires. Activate the
tracer by using it as a context manager.

In [ ]:
class HelloTracer(pyc.BaseTracer):
    @pyc.before_stmt
    def handle_stmt(self, *_, **__):
        print("Hello, world!")


with HelloTracer:
    # the loop body statement runs 10 times, plus the `for` statement itself
    pyc.exec("for _ in range(10): pass")

## 2. Handlers can *change* values, not just watch them

For many events the value a handler **returns replaces** the value of the
instrumented expression. Here every float literal is promoted to an exact
`Decimal`, so the classic `0.1 + 0.2` rounding surprise disappears:

In [ ]:
from decimal import Decimal


class ExactFloats(pyc.BaseTracer):
    @pyc.after_float
    def to_decimal(self, ret, *_, **__):
        return Decimal(str(ret))


with ExactFloats:
    pyc.exec("print(0.1 + 0.2)")   # -> 0.3   (not 0.30000000000000004)

## 3. Composability — just nest the tracers

Combining two `ast.NodeTransformer`s by hand is famously fiddly. In Pyccolo you
nest the context managers and the instrumentations layer automatically. Here the
assignment's right-hand side is first incremented, then doubled:

In [ ]:
class AddOne(pyc.BaseTracer):
    @pyc.after_assign_rhs
    def handle(self, ret, *_, **__):
        return ret + 1


class TimesTwo(pyc.BaseTracer):
    @pyc.after_assign_rhs
    def handle(self, ret, *_, **__):
        return ret * 2


with AddOne:
    with TimesTwo:
        env = pyc.exec("x = 42")

print(env["x"])   # (42 + 1) * 2 == 86

## 4. New syntax — optional chaining & nullish coalescing

Pyccolo can also add brand-new *syntax* by declaring an **augmentation**. The
bundled `optional_chaining` example adds three operators borrowed from other
languages:

- `a?.b` — short-circuit to `None` when `a` is `None`;
- `a.?b` — permissive access that yields `None` if attribute `b` is missing;
- `a ?? b` — "nullish coalescing": `a` unless it's `None`, else `b`.

In [ ]:
from pyccolo.examples.optional_chaining import ScriptOptionalChainer as OptionalChainer

with OptionalChainer:
    # `?.` short-circuits when the left side is None
    pyc.exec("user = None; print('optional:  ', user?.name)")
    # `.?` tolerates a missing attribute
    pyc.exec("obj = object(); print('permissive:', obj.?missing)")
    # `??` falls back only on None (note: '' and 0 are kept)
    print("nullish 1: ", pyc.eval("'' ?? 'default'"))
    print("nullish 2: ", pyc.eval("None ?? 'default'"))

## 5. Source-to-source: desugar and re-sugar

Because augmentations are real source transforms, you can use Pyccolo purely as
a **syntax translator** — no execution required. `pyc.transform` desugars new
syntax down to plain Python, and `pyc.untransform` recovers the sugar from an
AST:

In [ ]:
with OptionalChainer:
    desugared = pyc.transform("y = a?.b?.c")
    print("desugared:  ", desugared)

    tree = pyc.parse("y = a?.b?.c", instrument=False)
    print("re-sugared: ", pyc.untransform(tree))

## 6. `sys.settrace`-style call/return tracing

Alongside the AST events, Pyccolo exposes the two events Python's tracer already
knows about: `call` and `return_`. These fire on **ordinary in-cell code** — no
`pyc.exec` needed — so they're a drop-in way to watch the call stack:

In [ ]:
class StackTracer(pyc.BaseTracer):
    @pyc.call
    def on_call(self, *_, **__):
        print("-> push a frame")

    @pyc.return_
    def on_return(self, *_, **__):
        print("<- pop a frame")


with StackTracer:
    def f():
        def g():
            return 42
        return g()

    f()   # push f, push g, pop g, pop f

## 7. Concolic execution — solving for inputs that reach a branch

A more advanced example: the `concolic` tracer runs a function on *symbolic*
inputs, records the branch conditions along the way, and solves for inputs that
drive execution down new paths. Starting from `x = 0` it discovers the exact
`x = 6` needed to hit the `"jackpot"` branch — no z3 required (a small
brute-force solver ships as a fallback):

In [ ]:
from pyccolo.examples.concolic import explore


def needs_solver(x):
    y = x * 2
    if y == 12:
        if x > 5:
            return "jackpot"
        return "almost"   # unreachable: y == 12 forces x == 6 > 5
    return "nope"


for r in explore(needs_solver, {"x": 0}):
    print(f"x = {r.inputs['x']:>2}  ->  {r.retval!r}")

## 8. No `pyc.exec` needed — trace the notebook itself

Everything above routed code through `pyc.exec(...)`. In IPython or Jupyter you can
instead hand a tracer to the shell, and **every cell you run** gets instrumented.

Load the extension, then register a tracer. Registration takes effect on the *next*
cell, since this one has already been rewritten.

In [ ]:
%load_ext pyccolo
%pyccolo register HelloTracer

# equivalently, from Python: pyc.register_ipython_tracer(HelloTracer)

Now the tracer sees this cell's statements directly — no wrapper, no string of code.
Note that `Out[N]` still works: Pyccolo keeps the cell's trailing expression intact.

In [ ]:
for _ in range(3):
    pass

40 + 2

`%unload_ext pyccolo` undoes the patching and restores the shell. Like registration,
it takes effect on the *next* cell — this one was already rewritten before the magic
ran, so it still greets us once on its way out.

In [ ]:
%unload_ext pyccolo

In [ ]:
print("quiet again")
for _ in range(3):
    pass


## Where to go next

That's a tour — exact floats, composable tracers, optional chaining, source
translation, stack tracing, and concolic execution — all from one small handler
API. The bundled `pyccolo.examples` package has more: statement-level coverage,
`|>` pipeline operators, MacroPy-style quick lambdas and quasiquotes, lazy
imports, and implicit-async futures.

- 📖 [README & docs](https://github.com/smacke/pyccolo)
- 🧩 [Full event reference](https://pyccolo.readthedocs.io/)

**Edit any cell above and re-run it** to experiment — it all runs right here in
your browser.